In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json

# Define file paths (change these to match your file locations in your Google Drive)
correct_file_path = '/content/drive/MyDrive/gec_train_dataset_09-30_04-PM/shuffled_sampled_correct.txt'
incorrect_file_path = '/content/drive/MyDrive/gec_train_dataset_09-30_04-PM/shuffled_sampled_incorrect.txt'

# Function to read top 20% of lines from a file
def read_top_20_percent(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = [line.strip() for line in f if line.strip()]
    top_20_count = max(1, int(len(lines) * 0.2))  # Ensure at least 1 line
    return lines[:top_20_count]

# Read top 20% from both files
correct_lines = read_top_20_percent(correct_file_path)
incorrect_lines = read_top_20_percent(incorrect_file_path)

print(len(correct_lines), len(incorrect_lines))

# Ensure both lists are of equal length for pairing
min_length = min(len(correct_lines), len(incorrect_lines))
correct_lines = correct_lines[:min_length]
incorrect_lines = incorrect_lines[:min_length]

# Combine into desired JSON format
data = [
    {"incorrect_pair": incorrect, "correct_pair": correct}
    for incorrect, correct in zip(incorrect_lines, correct_lines)
]

# Save to JSON file
output_path = '/content/output_pairs.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

print(f"JSON saved to: {output_path}")

print(data[0])


254100 254100
✅ JSON saved to: /content/output_pairs.json
{'incorrect_pair': 'موجودہ حالات میں مسلمانوں کی جو دینی اور اخلاقی حالت ہے اور جس نوع کے سیاسی اور معاشی بحران سے ملت دوچار ہے ، اس کے پیش نظر مسلمانوں صفحات کے لیے طلاق کے معاملے میں صرف دو متبادل ہیں ۔', 'correct_pair': 'موجودہ حالات میں مسلمانوں کی جو دینی اور اخلاقی حالت ہے اور جس نوع کے سیاسی اور معاشی بحران سے ملت دوچار ہے، اس کے پیش نظر مسلمانوں کے لیے طلاق کے معاملے میں صرف دو متبادل ہیں۔'}


In [ ]:
%pip install torch transformers datasets evaluate tqdm

%pip install -q "transformers[sentencepiece]" datasets accelerate peft gdown bitsandbytes

%pip install deepspeed

%pip install evaluate

%pip install huggingface_hub[hf_xet]

%pip install hf_xet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 13.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 29.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 14.0 MB/s eta 0:00:00
  Created wheel for deepspeed: filename=deepspeed-0.17.6-py3-none-any.whl size=1740089 sha256=8c41f55a48337da21cb3ff1612246a88aa231b38085231294baad682f961e4f5
  Stored in directory: /root/.cache/pip/wheels/25/49/67/54ec3b6fa6f9dd03ba2af91e1e5c3d36fc2436261108d0860b
Successfully built deepspeed


In [5]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForCausalLM
)
from peft import PeftModel
from tqdm import tqdm
import json
import os


In [6]:
def load_data(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    incorrect = [item["incorrect_pair"] for item in data]
    correct = [item["correct_pair"] for item in data]
    return incorrect, correct


In [7]:
def load_model_and_tokenizer(base=None, checkpoint_dir=None, model_type=None):
    if base:
        tokenizer = AutoTokenizer.from_pretrained(base)
        if model_type == "seq2seq":
            model = AutoModelForSeq2SeqLM.from_pretrained(base)
        elif model_type == "causal":
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token
            model = AutoModelForCausalLM.from_pretrained(base)
        else:
            raise ValueError(f"Unknown model type for base {base}: {model_type}")
    else: # Loading from checkpoint_dir
        if model_type == "causal":
            tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir)
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token
            model = AutoModelForCausalLM.from_pretrained(checkpoint_dir) # Use checkpoint_dir here
        elif model_type == "seq2seq":
            tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir)
            model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint_dir)
        else:
            raise ValueError("When using --checkpoint_dir, you must specify --model_type (causal or seq2seq)")
    return model, tokenizer, model_type

In [8]:
def get_output_path(base=None, checkpoint_dir=None):
    os.makedirs("predictions", exist_ok=True)
    if base:
        checkpoint_name = base.replace("/", "_").replace("\\", "_")
        filename = f"{checkpoint_name}_predictions.json"
    else:
        checkpoint_dir = os.path.normpath(checkpoint_dir)
        parts = checkpoint_dir.split(os.sep)
        checkpoint_name = "_".join(parts[-2:]) if len(parts) >= 2 else parts[-1]
        checkpoint_name = checkpoint_name.replace("/", "_").replace("\\", "_")
        filename = f"{checkpoint_name}_predictions.json"

    return os.path.join("predictions", filename)


In [9]:
def generate_predictions(model, tokenizer, sources, model_type, max_len=128, batch_size=4):
    preds = []
    device = model.device
    for i in tqdm(range(0, len(sources), batch_size)):
        batch = sources[i:i + batch_size]
        if model_type == "seq2seq":
            inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True).to(device)
            with torch.no_grad():
                outputs = model.generate(**inputs, max_new_tokens=max_len)
            preds.extend(tokenizer.batch_decode(outputs, skip_special_tokens=True))
        else:  # causal (e.g., Alif / LLaMA)
            inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True).to(device)
            with torch.no_grad():
                outputs = model.generate(**inputs, max_new_tokens=max_len, do_sample=False)
            decoded = [tokenizer.decode(o, skip_special_tokens=True) for o in outputs]
            preds.extend([out.replace(inp, "").strip() for inp, out in zip(batch, decoded)])
    return preds


In [10]:
def run_experiments(arg_list):
    for args in arg_list:
        print("\nRunning experiment:", args)

        output_path = get_output_path(args.get("base"), args.get("checkpoint_dir"))
        print(f"Output will be saved to: {output_path}")

        # Load dataset
        incorrect, correct = load_data(args["data_file"])

        # Load model
        model, tokenizer, model_type = load_model_and_tokenizer(
            base=args.get("base"),
            checkpoint_dir=args.get("checkpoint_dir"),
            model_type=args.get("model_type")
        )
        model.eval()
        device = "cuda" if torch.cuda.is_available() else "cpu"
        model.to(device)
        print(f"Using device: {device}")

        # Generate predictions
        preds = generate_predictions(model, tokenizer, incorrect, model_type)

        # Save combined results
        combined = [
            {"incorrect": inc, "correct": cor, "predicted": pred}
            for inc, cor, pred in zip(incorrect, correct, preds)
        ]
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(combined, f, indent=2, ensure_ascii=False)
        print(f"Predictions saved to {output_path}")


In [11]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
 # List of runs
base_checkpt_dir = "/content/drive/MyDrive/checkpoints"
arg_list = [
    # --- Base Hugging Face Models ---
    # {"base": "bigscience/mt0-large", "data_file": "/content/output_pairs.json", "model_type": "seq2seq"},
    {"base": "facebook/nllb-200-3.3B", "data_file": "/content/output_pairs.json", "model_type": "seq2seq"},
    # {"base": "google/byt5-large", "data_file": "/content/output_pairs.json", "model_type": "seq2seq"},
    # {"base": "large-traversaal/Alif-1.0-8B-Instruct", "data_file": "/content/output_pairs.json", "model_type": "causal"},

    # --- Local Fine-tuned Checkpoints (Google Drive) ---
    # {"checkpoint_dir": f"{base_checkpt_dir}/mt0_checkpoint", "model_type": "seq2seq", "data_file": "/content/output_pairs.json"},
    # {"checkpoint_dir": f"{base_checkpt_dir}/byte_checkpoint", "model_type": "seq2seq", "data_file": "/content/output_pairs.json"},
    # {"checkpoint_dir": f"{base_checkpt_dir}/alif/checkpoint-10000", "model_type": "causal", "data_file": "/content/output_pairs.json"}
]

# Run all
run_experiments(arg_list)


Running experiment: {'base': 'facebook/nllb-200-3.3B', 'data_file': '/content/output_pairs.json', 'model_type': 'seq2seq'}
Output will be saved to: predictions/facebook_nllb-200-3.3B_predictions.json


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Using device: cuda


  9%|▉         | 5605/63525 [5:11:17<51:33:15,  3.20s/it]

In [ ]:
# Evaluate BLEU & BERTScore for all predicted files

import json
import os
import evaluate

def evaluate_all_predictions(pred_dir="predictions"):
    bleu = evaluate.load("bleu")
    bertscore = evaluate.load("bertscore")

    results = []

    for filename in os.listdir(pred_dir):
        if not filename.endswith(".json"):
            continue

        path = os.path.join(pred_dir, filename)
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        preds = [d["predicted"] for d in data]
        refs = [d["correct"] for d in data]

        print(f"\nEvaluating: {filename}")
        bleu_score = bleu.compute(predictions=preds, references=refs)
        bertscore_score = bertscore.compute(predictions=preds, references=refs, lang="ur")

        bleu_val = bleu_score["bleu"] * 100
        bert_f1_mean = sum(bertscore_score["f1"]) / len(bertscore_score["f1"])

        print(f"  BLEU Score: {bleu_val:.2f}")
        print(f"  BERTScore (F1 mean): {bert_f1_mean:.4f}")

        results.append({
            "file": filename,
            "BLEU": bleu_val,
            "BERTScore_F1": bert_f1_mean
        })

    print("\nEvaluation complete.")
    return results


# Run evaluation
all_results = evaluate_all_predictions()

# Optionally, save summary
with open("predictions/evaluation_summary.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2)

print("Summary saved to predictions/evaluation_summary.json")
